In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from app.core.parser import *
from app.core.loader import *
from app.reporting.history_report import *
from app.models.features import *
from app.models.train import *

import pandas as pd
import sqlite3
pd.set_option('future.no_silent_downcasting', True)


<class 'dict'>
{'pp': 'pred', 'op': 'order', 'rd': 'reldate', 'ys': 'year', 'ws': 'week', 'qs': 'qty', 'ds': 'idx', 'pi': 'predidx', 'oi': 'orderidx', 'py': 'predyear', 'oy': 'orderyear', 'pw': 'predweek', 'ow': 'orderweek', 'pq': 'predqty', 'oq': 'orderqty'}


/var/folders/4x/k1x7fbgd5bl08h5y4f2kbghw0000gn/T/ipykernel_19778/2987928993.py:11: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


In [4]:

#parent_dir = Path(__file__).resolve().parent
parent_dir = Path.cwd()
data = parent_dir / 'app' /  'data'

raw_dir = data / 'raw'
waterfall_dir =  raw_dir/ 'Waterfall'
consump_dir = raw_dir /  'Consumption'
cost_dir = raw_dir / 'Cost.txt'

proc_dir = data /  "Processed"

conn = sqlite3.connect(proc_dir / 'arlington.db')
cursor = conn.cursor()

# Loader behavior
sql_setup(conn) # doesn't do anything if db exists
waterfall_to_sql(conn, waterfall_dir=waterfall_dir)
consumption_to_sql(conn,  consump_dir=consump_dir)

## Push data from raw / cost.txt to sql 
cost_to_sql(conn, cost_dir=cost_dir)

# VERIFY DATA
list_all_tables_row_totals(conn) # print no. rows per table
loaded = get_loaded_files(conn, filter_type='waterfall')
all_wf= sorted(os.listdir(waterfall_dir))

# READ/PULL DATA
## Read within range 

# wf = filter_SQL(conn, 
#                 #idx_start=week_to_idx(2024,2), 
#                 idx_end=week_to_idx(2024, 2),  
#                 table='waterfall')

## Read all 
#test =  filter_SQL(conn, table='cost')

conn.close()

Bulk waterfall processing: 
     Skipping  {'.DS_Store'}  due to invalid data type
     Skipping  2024wk01.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk02.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk03.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk04.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk05.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk06.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk07.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk08.prn  identical copy processed; suppress this behavior by setting overwrite=True
     Skipping  2024wk09.prn  identical copy processed; suppress this behavior by sett

In [6]:
# TRAIN  
conn = sqlite3.connect(dirs['processed']/'arlington.db')

# Get all data
hist_wf = filter_SQL(conn, table='waterfall') # with releases/dates
wf = filter_SQL(conn, table='waterfall_agg') # prediction-wise rows  

cf = filter_SQL(conn, table='consumption') # 
cost= filter_SQL(conn, table='cost')

loaded = filter_SQL(conn, table='loaded_files')
conn.close() 


# create features in memory (TODO: save to DB- this has CAVEATs as features are a function of available info)

In [8]:
waterfall = waterfall_features(wf)

In [9]:
consumption = consumption_features(cf)

In [10]:
remove_nans = True
merged = waterfall.merge(consumption, on=['part', 'orderidx'], how='left').sort_values(["predidx", "part", "orderidx"])
print("Original merged size: ", merged.shape[0])
current_date = week_to_idx(2026, 5)
print("Current week (as idx): " , current_date)
real_data = merged[(merged.orderidx < current_date) # latest orders that "should" be consumed
                   &
                   (merged.orderidx >= cf.orderidx.min())] # earliest orders with consumption

if remove_nans:
    real_data = real_data[~real_data[oq].isna()]
else: # Infill zero where no order is given (risk: missing order data might be biasing model)
    real_data = real_data.fillna(0)

print("After filtering: ", real_data.shape[0])
real_data = pd.concat([real_data,id_features(real_data.part, 8)], axis=1)
real_data = real_data.merge(cost[['part', 'amount']], how='left', on='part')
# n_prior_pred(wf, 1)
# n_prior_pred(wf, 2)
# n_prior_pred(wf, 3)

Original merged size:  930642
Current week (as idx):  105357
After filtering:  368290


In [7]:
conn = sqlite3.connect(dirs['sql'])

list_all_tables_row_totals(conn)
conn.close()


waterfall (882818,)
consumption (111224,)
cost (9837,)
waterfall_agg (854582,)
loaded_files (194,)


In [94]:
from app.reporting.history_report import *


# Training process
remove_nans = True
conn = sqlite3.connect(dirs['sql'])

# Get all data
wf = filter_SQL(conn, table='waterfall_agg') # prediction-wise rows  
waterfall = waterfall_features(wf)
cf = filter_SQL(conn, table='consumption') # 
consumption = consumption_features(cf)
merged = waterfall.merge(consumption).sort_values(["predidx", "part", "orderidx"])
current_date = week_to_idx(2026, 5)
real_data = merged[(merged.orderidx < current_date) # latest orders that "should" be consumed
                   &
                   (merged.orderidx >= cf.orderidx.min())] # earliest orders with consumption

if remove_nans:
    real_data = real_data[~real_data[oq].isna()]
else: # Infill zero where no order is given (risk: missing order data might be biasing model)
    real_data = real_data.fillna(0)
real_data = pd.concat([real_data,id_features(real_data.part, 8)], axis=1)

conn.close() 


# create features in memory (TODO: save to DB- this has CAVEATs as features are a function of available info)


In [89]:
from app.models.train import *

In [29]:
df.columns

Index(['part', 'orderidx', 'predidx', 'predqty', 'pcr', 'lookahead_wk',
       'predqty_lag_qty', 'lag_ratio', 'predqty_mean_qty', 'predqty_std_qty',
       'predqty_cv', 'predqqty_max_qty', 'orderidx_part_count',
       'predqty_1wkdiff_qty', 'orderqty', 'roll2mean_orderqty_qty',
       'roll2std_orderqty_qty', 'roll3mean_orderqty_qty',
       'roll3std_orderqty_qty', 'part_lag1_qty', 'part_lag2_qty', 'pn0', 'pn1',
       'pn2', 'pn3', 'pn4', 'pn5', 'pn6', 'pn7', 'L', 'M', 'amount',
       'predVolume'],
      dtype='str')

In [ ]:
a = 'amount'
obj    = 'log'
loss   = 'absolute_error'
L2     =1.0 
N      = 1
df = real_data.copy()
real_qty = df[oq]
holdout_weeks=5

results_df = []
split_week = df[oi].max() - holdout_weeks

# TODO Verify if there is sufficient information in the remainder of the df (ie is (split_week - df[oi].min) > (df[oi].max - split-week))

df = df.sort_values(["part", oi]).copy()
df['predVolume'] = df[pq]*df[a]

train_mask = df[pre['oi']] < split_week
test_mask  = df[pre['oi']] >= split_week

target   = (df[oq] - df[pq]) * df[a] # Volume Error
baseline = np.zeros_like(df[pq])

y, thresh = transform_target(target, objective=obj)
X = df.drop(columns=[oq, 'part'])
X.columns = [str(c) for c in X.columns]

valid     = ~y.isna()
X_valid   = X[valid]
y_valid   = y[valid]
df_valid  = df[valid]
train_idx = train_mask[valid]
test_idx  = test_mask[valid]

X_train, y_train = X_valid[train_idx], y_valid[train_idx]
X_test,  y_test  = X_valid[test_idx],  y_valid[test_idx]

baseline_qty = (df_valid[pq])[test_idx]
baseline_test = (df_valid[pq] * df_valid[a])[test_idx]
pred_qty_test =  df_valid[pq][test_idx]
pred_amt_test =  df_valid[a][test_idx]

# ── boolean masks for the two splits ──────────────────────────────────────
mask_lo = (pred_qty_test.values <= N)  
mask_hi = ~mask_lo  

# ── Model A: trained & evaluated only on PredQty > N rows ─────────────────
train_hi = train_idx & (df_valid[pq] > N)
X_tr_A, y_tr_A = X_valid[train_hi], y_valid[train_hi]

mA = HistGradientBoostingRegressor(loss=loss, l2_regularization=L2)
mA.fit(X_tr_A, y_tr_A)

# ── Model B: trained & evaluated only on PredQty <= N rows ────────────────
train_lo = train_idx & (df_valid[pq] <= N)
X_tr_B, y_tr_B = X_valid[train_lo], y_valid[train_lo]

mB = HistGradientBoostingRegressor(loss=loss, l2_regularization=L2)
mB.fit(X_tr_B, y_tr_B)

# ── predict on test split ─────────────────────────────────────────────────
preds_A_raw = inverse_transform_target(mA.predict(X_test[mask_hi]), objective=obj)
preds_B_raw = inverse_transform_target(mB.predict(X_test[mask_lo]), objective=obj)
y_test_raw = inverse_transform_target(y_test.values, objective=obj)

preds_combined = np.empty(len(X_test))
preds_combined[mask_hi] = preds_A_raw
preds_combined[mask_lo] = preds_B_raw
predQty_combined = np.round(preds_combined/pred_amt_test)


In [27]:
baseline_test

305210    127.35000
305211      0.00000
311973    127.35000
318299    127.35000
324307    127.35000
            ...    
339893    761.25336
344309    815.62860
348320    380.62668
351970    380.62668
355157    380.62668
Length: 41582, dtype: float64

array([-127.35   ,    0.     , -127.35   , ..., -380.62668, -380.62668,
       -380.62668], shape=(41582,))

In [14]:
predQty_combined

305210    -0.0
305211     0.0
311973    -0.0
318299    -0.0
324307    -0.0
          ... 
339893   -13.0
344309   -18.0
348320    -9.0
351970    -7.0
355157    -7.0
Name: amount, Length: 41582, dtype: float64

In [ ]:
# Testing questions
# Model improvement
## 
# Can i improve the time horizon of predictions by increase zero materialization? 
## Take cases where and backforecast zeros ie prior to initial prediction
